# GeoVision-CLIP Cali — Situación 2 · Notebook 1
## Fine-tune RemoteCLIP ViT-B/32 sobre tiles Sentinel-2 (13 bandas) + textos pseudo-label

Arquitectura Stage 1 (referencia: `arquitecturas/c63b5ec6-d3f9-444e-99b2-9c44ab75d435.jpeg` — Late Fusion B).
Solo el encoder visual ViT y el text encoder se entrenan en este notebook.
La fusión con S5P numérico y el SAE se hacen en `02_sae_afe_afc.py`.

Salida: `clip_finetuned.pt` con MD5 reproducible.

Conceptos y referencias: `docs/conceptos/clip-y-remoteclip.md`.

In [ ]:
!pip install open_clip_torch huggingface_hub -q

In [ ]:
from __future__ import annotations
import os
import gc
import json
import time
import hashlib
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import open_clip
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

TILES_PATH = Path("/kaggle/input/datasets/edwardsx/geovision-tiles-sit2")
OUT_DIR = Path("/kaggle/working/sit2")
CKPT_DIR = OUT_DIR / "checkpoints"
FIG_DIR = OUT_DIR / "figuras"
for d in (OUT_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "ViT-B-32"
REMOTECLIP_REPO = "chendelong/RemoteCLIP"
REMOTECLIP_FILE = f"RemoteCLIP-{MODEL_NAME}.pt"

TILE_PX_IN = 64
TILE_PX_CLIP = 224
# N_BANDAS_INPUT se determina dinámicamente tras inspeccionar el dataset
# (dropeamos SCL porque es categórica, no reflectancia óptica)

EMB_VIT = 512
BATCH = 64
EPOCHS = 30
LR = 1e-5
WEIGHT_DECAY = 0.2
WARMUP_EPOCHS = 2
SPLIT_VAL = 0.06  

print(f"Hyperparams: BATCH={BATCH} EPOCHS={EPOCHS} LR={LR} WD={WEIGHT_DECAY}")
print(f"Salida: {OUT_DIR}")

## 1. Cargar tiles y meta

In [ ]:
npz_path = TILES_PATH / "tiles_train.npz"
meta_path = TILES_PATH / "tiles_meta.parquet"
assert npz_path.exists(), f"No existe {npz_path}"
assert meta_path.exists(), f"No existe {meta_path}"

with np.load(npz_path, allow_pickle=False) as npz:
    tiles_arr = npz["data"]
    bands_s2 = [str(b) for b in npz["bands"]] if "bands" in npz.files else None
meta = pd.read_parquet(meta_path)

print(f"tiles_arr: shape={tiles_arr.shape} dtype={tiles_arr.dtype}")
print(f"meta     : shape={meta.shape}")
print(f"bandas   : {bands_s2}")
print(meta["clase"].value_counts())

# SCL es categórica (clases SCL 0-11), no reflectancia. La dropeamos del input al ViT.
BANDAS_OPTICAS = [b for b in bands_s2 if b != "SCL"]
IDX_OPTICAS = np.array([bands_s2.index(b) for b in BANDAS_OPTICAS], dtype=np.int64)
N_BANDAS_INPUT = len(BANDAS_OPTICAS)
print(f"\nBandas ópticas usadas como input al ViT (sin SCL): {N_BANDAS_INPUT}")
print(f"  {BANDAS_OPTICAS}")

## 2. Traducción de textos español → inglés
Los textos del muestreo están en español por convención del proyecto. RemoteCLIP fue
entrenado en inglés, así que traducimos los 5 templates fijos en código antes de
tokenizar. Sin Google Translate ni dependencia externa.

In [ ]:
def template_ingles(row: pd.Series) -> str:
    clase = row["clase"]
    if clase == "contaminacion_alta_NO2":
        return f"Urban area with elevated NO2 concentration ({row['no2']:.2e} mol/m2), heavy vehicular traffic."
    if clase == "contaminacion_alta_SO2":
        return f"Industrial plume with elevated SO2 ({row['so2']:.2e} mol/m2), Yumbo-Acopi corridor."
    if clase == "ozono_anomalo":
        return f"Anomalous ozone concentration ({row['o3']:.2e} mol/m2), active photochemistry."
    if clase == "vegetacion_densa":
        return f"Dense vegetation, NDVI={row['ndvi']:.2f}, sugarcane or forest."
    if clase == "suelo_urbano":
        return f"Built-up urban land, NDVI={row['ndvi']:.2f}, high construction density."
    raise ValueError(f"Clase desconocida: {clase}")


meta = meta.copy()
meta["texto_en"] = meta.apply(template_ingles, axis=1)
print("Ejemplos:")
for c in meta["clase"].unique():
    print(f"  [{c}] {meta[meta.clase == c]['texto_en'].iloc[0]}")

## 3. Split train/val estratificado por clase

In [ ]:
rng = np.random.default_rng(SEED)
indices_train, indices_val = [], []
for c in meta["clase"].unique():
    idx_c = meta.index[meta["clase"] == c].to_numpy()
    rng.shuffle(idx_c)
    n_val = int(round(len(idx_c) * SPLIT_VAL))
    indices_val.extend(idx_c[:n_val])
    indices_train.extend(idx_c[n_val:])
indices_train = np.array(sorted(indices_train))
indices_val = np.array(sorted(indices_val))
print(f"train: {len(indices_train)}  | val: {len(indices_val)}")
print("val por clase:")
print(meta.loc[indices_val, "clase"].value_counts())

## 4. Descargar RemoteCLIP ViT-B/32

In [ ]:
print(f"Descargando {REMOTECLIP_FILE} desde HuggingFace {REMOTECLIP_REPO}...")
t0 = time.time()
ckpt_path = hf_hub_download(repo_id=REMOTECLIP_REPO, filename=REMOTECLIP_FILE)
print(f"  ok en {time.time()-t0:.1f}s | path: {ckpt_path}")

clip_model, _, _ = open_clip.create_model_and_transforms(MODEL_NAME)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
state_dict = torch.load(ckpt_path, map_location="cpu")
msg = clip_model.load_state_dict(state_dict)
print(f"  carga state_dict: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")

# Reset logit_scale a temperatura inicial sana (evita colapso)
with torch.no_grad():
    clip_model.logit_scale.fill_(np.log(1 / 0.07))  # ~2.659 → exp = 14.3
print(f"logit_scale reseteado: {clip_model.logit_scale.exp().item():.2f}")  # debe dar ~14.3

## 5. Adaptar primera conv 3ch → 13ch
Técnica documentada en `docs/conceptos/clip-y-remoteclip.md`:
- Copiamos pesos RGB originales en posiciones B4/B3/B2.
- Inicializamos las 10 bandas extra como `mean(RGB) * (3 / 13)` para preservar
magnitud de activación promedio.

In [ ]:
orig_conv = clip_model.visual.conv1
print(f"conv1 original: in={orig_conv.in_channels} out={orig_conv.out_channels}")
print(f"               kernel={orig_conv.kernel_size} stride={orig_conv.stride}")

idx_R_in = BANDAS_OPTICAS.index("B4")
idx_G_in = BANDAS_OPTICAS.index("B3")
idx_B_in = BANDAS_OPTICAS.index("B2")
print(f"Indices RGB en input ViT: R(B4)={idx_R_in} G(B3)={idx_G_in} B(B2)={idx_B_in}")

new_conv = nn.Conv2d(
    in_channels=N_BANDAS_INPUT,
    out_channels=orig_conv.out_channels,
    kernel_size=orig_conv.kernel_size,
    stride=orig_conv.stride,
    bias=False,
)
with torch.no_grad():
    w_orig = orig_conv.weight.data
    w_new = torch.zeros_like(new_conv.weight.data)
    w_new[:, idx_R_in] = w_orig[:, 0]
    w_new[:, idx_G_in] = w_orig[:, 1]
    w_new[:, idx_B_in] = w_orig[:, 2]
    rgb_mean = w_orig.mean(dim=1, keepdim=False)
    factor = 3.0 / N_BANDAS_INPUT
    for b in range(N_BANDAS_INPUT):
        if b not in (idx_R_in, idx_G_in, idx_B_in):
            w_new[:, b] = rgb_mean * factor
    new_conv.weight.copy_(w_new)
clip_model.visual.conv1 = new_conv
print(f"conv1 adaptada: in={new_conv.in_channels} out={new_conv.out_channels}")

## 6. Dataset y normalización

In [ ]:
S2_REFL_MAX = 10000.0

# Normalización por banda (reemplaza la división simple por S2_REFL_MAX)
flat = tiles_arr[:, IDX_OPTICAS, :, :].reshape(len(tiles_arr), N_BANDAS_INPUT, -1)
BAND_MEAN = flat.mean(axis=(0, 2)).astype(np.float32)  # (12,)
BAND_STD  = flat.std(axis=(0, 2)).astype(np.float32) + 1e-6
print(f"BAND_MEAN (primeras 4): {BAND_MEAN[:4].round(1)}")
print(f"BAND_STD  (primeras 4): {BAND_STD[:4].round(1)}")

class TilesDataset(Dataset):
    def __init__(self, indices: np.ndarray, tiles: np.ndarray, meta_df: pd.DataFrame, idx_opticas: np.ndarray):
        self.indices = indices
        self.tiles = tiles
        self.meta = meta_df.reset_index(drop=True)
        self.idx_opticas = idx_opticas

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        global_idx = int(self.indices[i])
        tile = self.tiles[global_idx][self.idx_opticas].astype(np.float32)
        # Normalización por banda (media/std del dataset completo)
        tile = (tile - BAND_MEAN[:, None, None]) / BAND_STD[:, None, None]
        tile = np.clip(tile, -3.0, 3.0)  # clip a ±3 sigmas
        x = torch.from_numpy(tile)
        x = F.interpolate(x.unsqueeze(0), size=TILE_PX_CLIP, mode="bilinear", align_corners=False).squeeze(0)
        texto = self.meta.iloc[global_idx]["texto_en"]
        return x, texto

def collate(batch):
    imgs = torch.stack([b[0] for b in batch], dim=0)
    txts = [b[1] for b in batch]
    return imgs, txts

train_ds = TilesDataset(indices_train, tiles_arr, meta, IDX_OPTICAS)
val_ds   = TilesDataset(indices_val,   tiles_arr, meta, IDX_OPTICAS)
# BUSCA ESTA PARTE EN TU CÓDIGO Y CAMBIALA:
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0, collate_fn=collate, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0, collate_fn=collate)
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")

## 7. Loop de fine-tune con InfoNCE bidireccional
Loss: `(CE(softmax(logits_per_image), I) + CE(softmax(logits_per_text), I)) / 2`.
Scheduler: cosine annealing con warmup 1 epoch.

In [ ]:
clip_model = clip_model.to(DEVICE)

# Reset logit_scale a temperatura inicial sana
with torch.no_grad():
    clip_model.logit_scale.fill_(np.log(1 / 0.07))
print(f"logit_scale inicial: {clip_model.logit_scale.exp().item():.2f}")

# Congela bloques 0-5 del visual y text encoder
for i in range(6):
    for p in clip_model.visual.transformer.resblocks[i].parameters():
        p.requires_grad = False
for i in range(6):
    for p in clip_model.transformer.resblocks[i].parameters():
        p.requires_grad = False

# Recolecta IDs de parámetros ya asignados para evitar duplicados
assigned_ids = set()

def get_params_unique(module_list):
    params = []
    for m in module_list:
        for p in m.parameters():
            if id(p) not in assigned_ids and p.requires_grad:
                assigned_ids.add(id(p))
                params.append(p)
    return params

# Grupo 1: bloques visuales 6-9
g1 = get_params_unique([clip_model.visual.transformer.resblocks[i] for i in range(6, 10)])

# Grupo 2: bloques visuales 10-11
g2 = get_params_unique([clip_model.visual.transformer.resblocks[i] for i in range(10, 12)])

# Grupo 3: bloques texto 6-11
g3 = get_params_unique([clip_model.transformer.resblocks[i] for i in range(6, 12)])

# Grupo 4: projection heads (visual.proj, text_projection, ln_final, ln_post)
proj_params = []
for name, p in clip_model.named_parameters():
    if id(p) not in assigned_ids and p.requires_grad:
        if any(k in name for k in ["proj", "text_projection", "ln_final", "ln_post"]):
            assigned_ids.add(id(p))
            proj_params.append(p)

# Grupo 5: logit_scale
logit_params = []
if id(clip_model.logit_scale) not in assigned_ids:
    assigned_ids.add(id(clip_model.logit_scale))
    logit_params.append(clip_model.logit_scale)

param_groups = [
    {"params": g1,           "lr": 2e-5,  "name": "visual_6-9"},
    {"params": g2,           "lr": 5e-5,  "name": "visual_10-11"},
    {"params": g3,           "lr": 5e-5,  "name": "text_6-11"},
    {"params": proj_params,  "lr": 1e-4,  "name": "projections"},
    {"params": logit_params, "lr": 1e-3,  "name": "logit_scale"},
]

# Muestra cuántos parámetros hay en cada grupo
for g in param_groups:
    n = sum(p.numel() for p in g["params"])
    print(f"  {g['name']:20s}: {n/1e6:.2f}M params @ lr={g['lr']:.0e}")

n_train = sum(p.numel() for g in param_groups for p in g["params"])
n_total = sum(p.numel() for p in clip_model.parameters())
print(f"Entrenables: {n_train/1e6:.1f}M / {n_total/1e6:.1f}M ({100*n_train/n_total:.1f}%)")

optimizer = torch.optim.AdamW(
    param_groups,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),
    eps=1e-6,
)

total_steps  = EPOCHS * len(train_loader)
warmup_steps = WARMUP_EPOCHS * len(train_loader)

def lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def infonce_loss(image_features, text_features, logit_scale):
    image_features = F.normalize(image_features, dim=-1)
    text_features  = F.normalize(text_features,  dim=-1)
    logits   = logit_scale * image_features @ text_features.t()
    targets  = torch.arange(logits.size(0), device=logits.device)
    loss_i   = F.cross_entropy(logits,    targets)
    loss_t   = F.cross_entropy(logits.t(), targets)
    return (loss_i + loss_t) / 2, logits

def recall_at_k(logits: torch.Tensor, k_values=(1, 5, 10)) -> dict[int, float]:
    n       = logits.size(0)
    targets = torch.arange(n, device=logits.device)
    ranks   = logits.argsort(dim=-1, descending=True)
    out = {}
    for k in k_values:
        topk = ranks[:, :k]
        hits = (topk == targets.unsqueeze(1)).any(dim=1).float().mean().item()
        out[k] = hits
    return out

@torch.no_grad()
def evaluar(loader, model, tokenizer_):
    model.eval()
    all_img, all_txt = [], []
    val_losses = [] # <--- Nueva lista para pérdida
    
    for imgs, txts in loader:
        imgs = imgs.to(DEVICE)
        toks = tokenizer_(txts).to(DEVICE)
        
        img_f = model.encode_image(imgs)
        txt_f = model.encode_text(toks)
        
        # CÁLCULO DE VAL LOSS
        loss, _ = infonce_loss(img_f, txt_f, model.logit_scale.exp())
        val_losses.append(loss.item())
        
        all_img.append(F.normalize(img_f, dim=-1))
        all_txt.append(F.normalize(txt_f, dim=-1))
        
    img_e  = torch.cat(all_img)
    txt_e  = torch.cat(all_txt)
    logits = img_e @ txt_e.t()
    rec    = recall_at_k(logits)
    
    return rec, np.mean(val_losses) # <--- Ahora devuelve ambos

# DEFINE LAS RUTAS (Asegúrate de que esto esté antes del loop)
OUT_DIR = Path("/kaggle/working/sit2")
CKPT_DIR = OUT_DIR / "checkpoints"
best_path = CKPT_DIR / "clip_finetuned_best.pt" # <--- ESTA ES LA QUE FALTA
log_path = OUT_DIR / "train_log.txt"

# Asegurar que las carpetas existan
CKPT_DIR.mkdir(parents=True, exist_ok=True)

hist_train_loss = []
hist_val_loss = []
hist_val_recall = {1: [], 5: [], 10: []}
best_r5 = -1.0

# Asegurar que el directorio de salida existe
CKPT_DIR.mkdir(parents=True, exist_ok=True)

t_start = time.time()
for epoch in range(EPOCHS):
    # --- FASE DE ENTRENAMIENTO NORMAL ---
    clip_model.train()
    epoch_train_losses = []
    pbar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}")
    
    for imgs, txts in pbar:
        imgs = imgs.to(DEVICE)
        toks = tokenizer(txts).to(DEVICE)
        
        # Forward pass
        img_f = clip_model.encode_image(imgs)
        txt_f = clip_model.encode_text(toks)
        
        # Calculo de perdida InfoNCE de entrenamiento
        loss, _ = infonce_loss(img_f, txt_f, clip_model.logit_scale.exp())
        
        # Optimizacion
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in clip_model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        scheduler.step()
        
        # Clamp logit_scale segun buenas practicas de CLIP
        with torch.no_grad():
            clip_model.logit_scale.clamp_(np.log(1), np.log(100))
        
        epoch_train_losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(epoch_train_losses[-20:]):.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    # --- FASE DE VALIDACION (AGREGADA) ---
    # Llamamos a evaluar para obtener metricas y perdida de validacion
    mean_train_loss = float(np.mean(epoch_train_losses))
    rec, mean_val_loss = evaluar(val_loader, clip_model, tokenizer)
    
    # Guardar en historico para las graficas
    hist_train_loss.append(mean_train_loss)
    hist_val_loss.append(mean_val_loss)
    for k in (1, 5, 10):
        hist_val_recall[k].append(rec[k])

    # Imprimir resumen de la epoca
    cur_lr = scheduler.get_last_lr()[0]
    print(f"  Resultados Epoch {epoch+1}:")
    print(f"  Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f}")
    print(f"  Val R@1: {rec[1]:.3f} | Val R@5: {rec[5]:.3f} | Val R@10: {rec[10]:.3f}")

    # Guardar el mejor modelo basado en R@5
    if rec[5] > best_r5:
        best_r5 = rec[5]
        torch.save(clip_model.state_dict(), best_path)
        print(f"  --- Checkpoint guardado (Nuevo mejor R@5: {best_r5:.3f}) ---")

print(f"\nEntrenamiento finalizado en {(time.time() - t_start) / 60:.1f} minutos")

## 8. Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

# 1. Configurar el estilo y el lienzo (2 subplots: Perdidas y Recalls)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Determinar el numero de epocas reales ejecutadas
epochs_run = len(hist_train_loss)
x_range = range(1, epochs_run + 1)

# --- GRAFICA 1: CONVERGENCIA DE FUNCIONES DE PERDIDA ---
axes[0].plot(x_range, hist_train_loss, color="#c44e52", lw=2.5, label="Train Loss (InfoNCE)")
axes[0].plot(x_range, hist_val_loss, color="#4c72b0", lw=2.5, linestyle='--', label="Val Loss (InfoNCE)")

axes[0].set_xlabel("Epoca", fontsize=12)
axes[0].set_ylabel("Perdida (Loss)", fontsize=12)
axes[0].set_title("Convergencia del Modelo: Train vs Validation Loss", fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- GRAFICA 2: METRICAS DE RENDIMIENTO (RECALL@K) ---
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]
for k, color in zip((1, 5, 10), colors):
    axes[1].plot(x_range, hist_val_recall[k], color=color, lw=2.5, label=f"Recall@{k}")

# Resaltar el mejor punto de Recall@5
best_r5_val = max(hist_val_recall[5])
best_epoch = hist_val_recall[5].index(best_r5_val) + 1
axes[1].annotate(f'Mejor R@5: {best_r5_val:.3f}', 
                 xy=(best_epoch, best_r5_val), 
                 xytext=(best_epoch + 2, best_r5_val + 0.02),
                 arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

axes[1].set_xlabel("Epoca", fontsize=12)
axes[1].set_ylabel("Puntuacion Recall", fontsize=12)
axes[1].set_title("Metricas de Recuperacion (Recall@K) en Validacion", fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Ajustar diseño y guardar
plt.tight_layout()
final_fig_path = FIG_DIR / "geovision_clip_training_results.png"
fig.savefig(final_fig_path, bbox_inches="tight", dpi=150)
plt.show()

print(f"Grafica final guardada con exito en: {final_fig_path}")

## 9. Checkpoint final y MD5 reproducible

In [ ]:
final_path = CKPT_DIR / "clip_finetuned.pt"
state = torch.load(best_path, map_location="cpu")
torch.save(state, final_path)


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


md5_ckpt = md5_file(final_path)
print(f"checkpoint final : {final_path}")
print(f"tamaño           : {final_path.stat().st_size / 1024**2:.1f} MB")
print(f"MD5              : {md5_ckpt}")
print(f"mejor val R@5   : {best_r5:.3f}")

manifest = {
    "model_name": MODEL_NAME,
    "remoteclip_source": f"{REMOTECLIP_REPO}/{REMOTECLIP_FILE}",
    "bandas_opticas": BANDAS_OPTICAS,
    "n_bandas_input": N_BANDAS_INPUT,
    "scl_dropeada": "SCL" in bands_s2,
    "tile_px_in": TILE_PX_IN,
    "tile_px_clip": TILE_PX_CLIP,
    "batch": BATCH,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "split_val": SPLIT_VAL,
    "seed": SEED,
    "best_val_recall_at_5": best_r5,
    "md5_checkpoint": md5_ckpt,
}
manifest_path = OUT_DIR / "manifest_stage1.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"manifest         : {manifest_path}")

## 10. Subir a Kaggle como dataset
Descomentar para subir tras validar.

In [ ]:
import shutil
UP = OUT_DIR / "upload"
UP.mkdir(exist_ok=True)
shutil.copy(final_path, UP / "clip_finetuned.pt")
shutil.copy(manifest_path, UP / "manifest_stage1.json")
shutil.copy(FIG_DIR / "geovision_clip_training_results.png", UP / "geovision_clip_training_results.png")
shutil.copy(log_path, UP / "train_log.txt")

(UP / "dataset-metadata.json").write_text(json.dumps({
    "title": "GeoVision Sit2 CLIP finetuned",
    "id": "edwardsx/geovision-sit2-clip",
    "licenses": [{"name": "CC-BY-SA-4.0"}],
}, indent=2))
print(f"Listo en {UP}: {[p.name for p in UP.iterdir()]}")
!kaggle datasets create -p /kaggle/working/sit2/upload  # primera vez
!kaggle datasets version -p /kaggle/working/sit2/upload -m "stage1 clip finetuned"

## 11. Evaluación zero-shot por clase

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.neighbors import KNeighborsClassifier

PROTOTIPOS = {
    "contaminacion_alta_NO2": [
        "Urban area with elevated NO2 concentration, heavy vehicular traffic.",
        "Dense urban land with high nitrogen dioxide pollution from vehicles.",
        "City center with NO2 plume from traffic emissions.",
    ],
    "contaminacion_alta_SO2": [
        "Industrial plume with elevated SO2, Yumbo-Acopi corridor.",
        "Heavy industry with sulfur dioxide emissions.",
        "Industrial district with SO2 plumes from factories.",
    ],
    "ozono_anomalo": [
        "Anomalous ozone concentration, active photochemistry.",
        "High tropospheric ozone from photochemical reactions.",
        "Elevated O3 levels under intense solar radiation.",
    ],
    "vegetacion_densa": [
        "Dense vegetation, sugarcane or forest.",
        "Lush green vegetation, cropland or natural forest.",
        "Dense canopy vegetation with high NDVI.",
    ],
    "suelo_urbano": [
        "Built-up urban land, high construction density.",
        "Dense city with concrete and rooftops.",
        "Urban built-up area with buildings and roads.",
    ],
}
CLASES = list(PROTOTIPOS.keys())

clip_model.load_state_dict(torch.load(best_path, map_location="cpu"))
clip_model = clip_model.to(DEVICE).eval()


@torch.no_grad()
def encode_imgs(indices):
    embs = []
    for i in range(0, len(indices), 64):
        batch = tiles_arr[indices[i:i + 64]][:, IDX_OPTICAS].astype(np.float32)
        batch = (batch - BAND_MEAN[None, :, None, None]) / BAND_STD[None, :, None, None]
        batch = np.clip(batch, -3.0, 3.0)
        x = torch.from_numpy(batch).to(DEVICE)
        x = F.interpolate(x, size=TILE_PX_CLIP, mode="bilinear", align_corners=False)
        embs.append(F.normalize(clip_model.encode_image(x), dim=-1).cpu())
    return torch.cat(embs)


@torch.no_grad()
def encode_prototipos():
    out = []
    for c in CLASES:
        toks = tokenizer(PROTOTIPOS[c]).to(DEVICE)
        emb = F.normalize(clip_model.encode_text(toks), dim=-1).mean(dim=0)
        out.append(F.normalize(emb, dim=-1))
    return torch.stack(out).cpu()


emb_val = encode_imgs(indices_val)
emb_proto = encode_prototipos()
pred = (emb_val @ emb_proto.t()).argmax(dim=-1).numpy()
pred_clase = np.array([CLASES[i] for i in pred])
real_clase = meta.loc[indices_val, "clase"].to_numpy()
acc_zs = (pred_clase == real_clase).mean()

print(f"Zero-shot accuracy: {acc_zs:.3f}  (chance=0.20, mejora={acc_zs/0.20:.1f}x)")
print(classification_report(real_clase, pred_clase, target_names=CLASES, digits=3))

## 12. Confusion matrix

In [ ]:
cm = confusion_matrix(real_clase, pred_clase, labels=CLASES)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm_norm, cmap="Blues", aspect="auto")
short = [c.replace("contaminacion_alta_", "") for c in CLASES]
ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(short, rotation=30, ha="right")
ax.set_yticklabels(short)
ax.set_xlabel("Predicción")
ax.set_ylabel("Real")
ax.set_title(f"Confusion matrix — acc={acc_zs:.3f}")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.0%}", ha="center", va="center",
                fontsize=9, color="white" if cm_norm[i, j] > 0.5 else "black")
plt.tight_layout()
fig.savefig(str(FIG_DIR / "clip_confusion_matrix.png"), bbox_inches="tight", dpi=130)
plt.show()

## 13. k-NN sobre embeddings

In [ ]:
indices_train_all = np.array([i for i in range(len(meta)) if i not in set(indices_val.tolist())])
emb_train = encode_imgs(indices_train_all)
y_train = meta.loc[indices_train_all, "clase"].to_numpy()

knn_acc = {}
for k in (1, 5, 11):
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(emb_train.numpy(), y_train)
    knn_acc[k] = float((knn.predict(emb_val.numpy()) == real_clase).mean())
    print(f"k-NN k={k:>2}: acc={knn_acc[k]:.3f}")

## 14. Recall@K agrupado por clase

In [ ]:
textos_val = [PROTOTIPOS[meta.iloc[int(i)]["clase"]][0] for i in indices_val]
with torch.no_grad():
    emb_txt = F.normalize(clip_model.encode_text(tokenizer(textos_val).to(DEVICE)), dim=-1).cpu()
sims = emb_val @ emb_txt.t()

recall_clase = {}
for k in (1, 5, 10):
    top = sims.argsort(dim=-1, descending=True)[:, :k].numpy()
    recall_clase[k] = float(np.mean([(real_clase[top[i]] == real_clase[i]).any() for i in range(len(real_clase))]))
    print(f"R@{k} (por clase): {recall_clase[k]:.3f}")

## 15. Veredicto

In [ ]:
veredicto = {
    "zero_shot_accuracy": float(acc_zs),
    "improvement_over_chance": float(acc_zs / 0.20),
    "knn_accuracy": knn_acc,
    "recall_at_k_por_clase": recall_clase,
    "report": classification_report(real_clase, pred_clase, target_names=CLASES, output_dict=True, digits=3),
}
(OUT_DIR / "eval_v2.json").write_text(json.dumps(veredicto, indent=2, default=str))

if acc_zs >= 0.40:
    print(f"OK acc={acc_zs:.3f} >= 0.40 -> pasar a Stage 2 (SAE)")
elif acc_zs >= 0.30:
    print(f"MARGINAL acc={acc_zs:.3f} -> evaluar reentreno")
else:
    print(f"BAJA acc={acc_zs:.3f} -> reentrenar")